In [1]:
from sentence_transformers import SentenceTransformer

ImportError: tokenizers>=0.19,<0.20 is required for a normal functioning of this module, but found tokenizers==0.21.1.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

In [ ]:

from sentence_transformers.models import StaticEmbedding
from datasets import load_dataset

c:\Users\david\Documents\git\SemanticSearch\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'cached_download' from 'huggingface_hub' (c:\Users\david\Documents\git\SemanticSearch\venv\Lib\site-packages\huggingface_hub\__init__.py)

In [ ]:


import duckdb
from typing import List

import os
import time
def play_chimes():
    sound_path = r"C:\Windows\Media\chimes.wav"
    os.system(f'powershell -c (New-Object Media.SoundPlayer "{sound_path}").PlaySync();')

ImportError: tokenizers>=0.19,<0.20 is required for a normal functioning of this module, but found tokenizers==0.21.1.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

In [ ]:
static_embedding = StaticEmbedding.from_model2vec("minishlab/potion-base-8M")
model = SentenceTransformer(modules=[static_embedding])

model.safetensors:   0%|          | 0.00/30.2M [00:00<?, ?B/s]

c:\Users\david\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\david\.cache\huggingface\hub\models--minishlab--potion-base-8M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md:   0%|          | 0.00/271k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/684k [00:00<?, ?B/s]

In [4]:
ds = load_dataset("ai-blueprint/fineweb-bbc-news")

Generating train split:   0%|          | 0/352549 [00:00<?, ? examples/s]

We can now create embeddings for the dataset. Normally, we might want to chunk our data into smaller batches to avoid losing precision, but for this example, we will just create embeddings for the full text of the dataset.

In [ ]:
def create_embeddings(batch):
    embeddings = model.encode(batch["text"], convert_to_numpy=True)
    batch["embeddings"] = embeddings.tolist()
    return batch

ds = ds.map(create_embeddings, batched=True)

Map:   0%|          | 0/352549 [00:00<?, ? examples/s]

In [7]:
def similarity_search_without_duckdb_index(
    query: str,
    k: int = 5,
    dataset_name: str = "ai-blueprint/fineweb-bbc-news-embeddings",
    embedding_column: str = "embeddings",
):
    # Use same model as used for indexing
    query_vector = model.encode(query)
    embedding_dim = model.get_sentence_embedding_dimension()

    sql = f"""
        SELECT 
            *,
            array_cosine_distance(
                {embedding_column}::float[{embedding_dim}], 
                {query_vector.tolist()}::float[{embedding_dim}]
            ) as distance
        FROM 'hf://datasets/{dataset_name}/**/*.parquet'
        ORDER BY distance
        LIMIT {k}
    """
    return duckdb.sql(sql).to_df()


similarity_search_without_duckdb_index("What is the future of AI?")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,url,text,embeddings,distance
0,https://www.bbc.com/news/technology-51064369,The last decade was a big one for artificial i...,"[-0.9206902980804443, 0.783940315246582, -2.00...",0.281200
1,http://www.bbc.com/news/technology-25000756,Singularity: The robots are coming to steal ou...,"[-1.5080468654632568, 0.7840216755867004, -1.1...",0.365842
2,http://www.bbc.co.uk/news/technology-25000756,Singularity: The robots are coming to steal ou...,"[-1.5080468654632568, 0.7840216755867004, -1.1...",0.365842
3,https://www.bbc.co.uk/news/technology-37494863,"Google, Facebook, Amazon join forces on future...","[-0.38261985778808594, 1.6644303798675537, -1....",0.380820
4,https://www.bbc.co.uk/news/technology-37494863,"Google, Facebook, Amazon join forces on future...","[-0.38261985778808594, 1.6644303798675537, -1....",0.380820


In [19]:
def _setup_vss():
    duckdb.sql(
        query="""
        INSTALL vss;
        LOAD vss;
        """
    )


def _drop_table(table_name):
    duckdb.sql(
        query=f"""
        DROP TABLE IF EXISTS {table_name};
        """
    )


def _create_table(dataset_name, table_name, embedding_column):
    duckdb.sql(
        query=f"""
        CREATE TABLE {table_name} AS 
        SELECT *, {embedding_column}::float[{model.get_sentence_embedding_dimension()}] as {embedding_column}_float 
        FROM 'hf://datasets/{dataset_name}/**/*.parquet';
        """
    )


def _create_index(table_name, embedding_column):
    duckdb.sql(
        query=f"""
        CREATE INDEX my_hnsw_index ON {table_name} USING HNSW ({embedding_column}_float) WITH (metric = 'cosine');
        """
    )


def create_index(dataset_name, table_name, embedding_column):
    _setup_vss()
    _drop_table(table_name)
    _create_table(dataset_name, table_name, embedding_column)
    _create_index(table_name, embedding_column)


create_index(
    dataset_name="ai-blueprint/fineweb-bbc-news-embeddings",
    table_name="fineweb_bbc_news_embeddings",
    embedding_column="embeddings",
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [20]:
def similarity_search_with_duckdb_index(
    query: str, k: int = 5, table_name: str = "fineweb_bbc_news_embeddings", embedding_column: str = "embeddings"
):
    embedding = model.encode(query).tolist()
    return duckdb.sql(
        query=f"""
        SELECT *, array_cosine_distance({embedding_column}_float, {embedding}::FLOAT[{model.get_sentence_embedding_dimension()}]) as distance 
        FROM {table_name}
        ORDER BY distance 
        LIMIT {k};
    """
    ).to_df()


similarity_search_with_duckdb_index("What is love?")

,url,text,embeddings,embeddings_float,distance
0,http://news.bbc.co.uk/cbbcnews/hi/newsid_17700...,February 14 is Valentine's Day.\nIt's named af...,"[-1.6512084007263184, -0.2739017605781555, -1....","[-1.6512084, -0.27390176, -1.5980083, 2.005432...",0.286753
1,http://news.bbc.co.uk/cbbcnews/hi/newsid_17700...,February 14 is Valentine's Day.\nIt's named af...,"[-1.6512084007263184, -0.2739017605781555, -1....","[-1.6512084, -0.27390176, -1.5980083, 2.005432...",0.286753
2,https://www.bbc.co.uk/news/magazine-24223786,A Point of View: Putting a price on love\nOur ...,"[-3.3412487506866455, 0.11048950254917145, -1....","[-3.3412488, 0.1104895, -1.4662294, 0.5659724,...",0.450999
3,https://www.bbc.com/news/in-pictures-43060122,"Your pictures: Valentine's Day\nEach week, we ...","[0.03464483842253685, -0.9412625432014465, -1....","[0.03464484, -0.94126254, -1.2021598, 0.427660...",0.468012
4,http://www.bbc.co.uk/news/magazine-24379830,Why is a children's book about rabbits being r...,"[-1.4426814317703247, -0.7511618733406067, -1....","[-1.4426814, -0.7511619, -1.336863, 1.4392253,...",0.476307
